# Text Classification Without Language Models

##### Author: [Radoslav Neychev](https://www.linkedin.com/in/radoslav-neychev/), telegram: [@rads_ai](https://t.me/rads_ai)

In this assignment, we will work on the task of sequence (text) classification using various word vectorization methods.

In [ ]:
# do not change the code in the block below
# __________start of block__________
import json
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython import display
from nltk.tokenize import WordPunctTokenizer
from sklearn import naive_bayes
from sklearn.base import BaseEstimator
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from torch import nn
from torch.nn import functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR

%matplotlib inline


out_dict = dict()
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# __________end of block__________

### Text Preprocessing and Tokenization

Preprocessing is almost identical to what was covered in the previous lesson. The `nltk` library [link](https://www.nltk.org) is widely used for text processing. You can find its detailed description and documentation at the link above.

In [ ]:
# do not change the code in the block below
# __________start of block__________
df = pd.read_csv(
    "https://github.com/clairett/pytorch-sentiment-classification/raw/master/data/SST2/train.tsv",
    delimiter="\t",
    header=None,
)
texts_train = df[0].values[:5000]
y_train = df[1].values[:5000]
texts_test = df[0].values[5000:]
y_test = df[1].values[5000:]


tokenizer = WordPunctTokenizer()
preprocess = lambda text: " ".join(tokenizer.tokenize(text.lower()))

text = 'How to be a grown-up at work: replace "I don\'t want to do that" with "Ok, great!".'
print(
    "before:",
    text,
)
print(
    "after:",
    preprocess(text),
)

texts_train = [preprocess(text) for text in texts_train]
texts_test = [preprocess(text) for text in texts_test]

# Small check that everything is done properly
assert (
    texts_train[5]
    == "campanella gets the tone just right funny in the middle of sad in the middle of hopeful"
)
assert texts_test[74] == "poetry in motion captured on film"
assert len(texts_test) == len(y_test)
# __________end of block__________

The following functions will help you visualize the network training process.

In [ ]:
# do not change the code in the block below
# __________start of block__________


def plot_train_process(
    train_loss, val_loss, train_accuracy, val_accuracy, title_suffix=""
):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    axes[0].set_title(" ".join(["Loss", title_suffix]))
    axes[0].plot(train_loss, label="train")
    axes[0].plot(val_loss, label="validation")
    axes[0].legend()

    axes[1].set_title(" ".join(["Validation accuracy", title_suffix]))
    axes[1].plot(train_accuracy, label="train")
    axes[1].plot(val_accuracy, label="validation")
    axes[1].legend()
    plt.show()


def visualize_and_save_results(
    model, model_name, X_train, X_test, y_train, y_test, out_dict
):
    for data_name, X, y, model in [
        ("train", X_train, y_train, model),
        ("test", X_test, y_test, model),
    ]:
        if isinstance(model, BaseEstimator):
            proba = model.predict_proba(X)[:, 1]
        elif isinstance(model, nn.Module):
            proba = model(X).detach().cpu().numpy()[:, 1]
        else:
            raise ValueError("Unrecognized model type")

        auc = roc_auc_score(y, proba)

        out_dict[f"{model_name}_{data_name}"] = auc
        plt.plot(*roc_curve(y, proba)[:2], label="{} AUC={:.4f}".format(data_name, auc))

    plt.plot(
        [0, 1],
        [0, 1],
        "--",
        color="black",
    )
    plt.legend(fontsize="large")
    plt.title(model_name)
    plt.grid()
    return out_dict


# __________end of block__________

## Review: Basic Concepts in Deep Learning

__Layer__ – a transformation applied to the source data. The simplest example is a linear layer, which is a linear transformation of the input data (i.e., just the transformation $WX + b$, as in linear regression).

__Activation function__ – a non-linear transformation applied to all input data element-wise. Thanks to activation functions, neural networks perform *non-linear transformations* on the initial features, thereby generating a more informative feature description.

__Neural network__ – a composition of linear and non-linear transformations (usually represented as a sequence of layers and activation functions). Deep Learning is dedicated to studying neural networks and their applicability to various tasks. In most cases, the entire neural network is one complex *differentiable* function, which imposes restrictions on the possibility of using certain transformations.

__Regularization__ – a mechanism for imposing constraints on the solution based on expert knowledge and/or a priori assumptions about the problem being solved. It can be represented as an additional term in the loss function (e.g., $L1$ or $L2$ regularization), as constraints on the model structure (`Dropout`, `Batch Normalization`), and in other forms.

__Loss function__ – a loss function that evaluates the quality of the prediction. Generally, the loss function is required to be differentiable (since network parameters are tuned using the *backpropagation* method). In some cases (e.g., in reinforcement learning), non-differentiable loss/reward functions are used. Their use requires modifications to the neural network training mechanism.

## Working with Texts and Sequences

* __Sequences__. Data – sets of values with a defined order relation. Values can be discrete (e.g., DNA) or can take values from a continuous interval (e.g., a data center's power consumption time series). Rearranging values results in a loss of information. The order relation must not be violated (testing on the past, training on the future).

* __Texts__. Data – sets of words/characters. In fact, they are sequences of values from a finite alphabet, but they have a fairly strict internal structure due to the existence of grammar.

In natural language processing (primarily in the form of text) and sequence modeling, recurrent networks and networks based on the attention mechanism have proven highly effective, such as the Transformer model proposed in the 2017 paper "Attention is all you need." Both recurrent and transformer-like models take into account the dependencies between sequence elements (and the presence of order in general), allowing them to automatically generate informative feature representations (similar to convolutional networks when working with images).

### Task #1. Bag of Words (BoW).

Use the classical approach to text vectorization: Bag of Words. You can either use `CountVectorizer` from `sklearn` or implement your own version.

Bag of Words assigns a unique index (word number in the vocabulary) to each word from the vocabulary and builds the final vector for the text as a set of counts for each word from the vocabulary. This approach is equivalent to building the sum of `one-hot` vectors for each of the words in the text.

#### __One-hot Encoding__.
Each word in the language can be associated with a unique index, and a vector can be assigned to the word where zeros are in all places except for the given index. This approach is called one-hot encoding. An example of such encoding can be seen below.

*Example: the word "dog" is in third place in a vocabulary of 5 words. Then it will correspond to the vector `[0, 0, 1, 0, 0]`. The word "cat" is in second place, it corresponds to the vector `[0, 1, 0, 0, 0]`. The word "kitten" is in fourth place, it corresponds to the vector `[0, 0, 0, 1, 0]`.*



Please note that in part 1, only the `k` most frequent words from the training part of the dataset are used.

In [ ]:
# do not change the code in the block below
# __________start of block__________

k = min(10000, len(set(" ".join(texts_train).split())))

counts = Counter(" ".join(texts_train).split())

bow_vocabulary = [key for key, val in counts.most_common(k)]


def text_to_bow(text):
    """convert text string to an array of token counts. Use bow_vocabulary."""
    sent_vec = np.zeros(len(bow_vocabulary))
    counts = Counter(text.split())
    for i, token in enumerate(bow_vocabulary):
        if token in counts:
            sent_vec[i] = counts[token]
    return np.array(sent_vec, "float32")


X_train_bow = np.stack(list(map(text_to_bow, texts_train)))
X_test_bow = np.stack(list(map(text_to_bow, texts_test)))

# Small check that everything is done properly if you are using local bow implementation
k_max = len(set(" ".join(texts_train).split()))
assert X_train_bow.shape == (len(texts_train), min(k, k_max))
assert X_test_bow.shape == (len(texts_test), min(k, k_max))
assert np.all(
    X_train_bow[5:10].sum(-1) == np.array([len(s.split()) for s in texts_train[5:10]])
)
assert len(bow_vocabulary) <= min(k, k_max)
assert X_train_bow[65, bow_vocabulary.index("!")] == texts_train[65].split().count("!")


bow_model = LogisticRegression(max_iter=1500).fit(X_train_bow, y_train)

out_dict = visualize_and_save_results(
    bow_model, "bow_log_reg_sklearn", X_train_bow, X_test_bow, y_train, y_test, out_dict
)
# __________end of block__________

The results are not bad, but overfitting is clearly visible. This conclusion can be drawn from the significant superiority of quality (AUC ROC) on the train set relative to the test set. Moreover, on the training set, the quality tends towards unity, while on the holdout set, it is significantly lower, meaning the model has caught many dependencies specific only to the training set. The problem of overfitting was basically discussed in previous lessons and will be encountered many more times in the course.

We will deal with overfitting in this task later. For now, implement a solution based on logistic regression, but using PyTorch. As a result, you should have a trained model that predicts probabilities for two classes. The quality on the test set should not be inferior to logistic regression.

In [ ]:
model = # your code here

Don't forget about loss functions: `nn.CrossEntropyLoss` combines `LogSoftMax` and `NLLLoss`. Also, don't forget the need to move tensors to the `device` being used.

In [ ]:
loss_function = # your code here
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
X_train_bow_torch = # your code here
X_test_bow_torch = # your code here

y_train_torch = # your code here
y_test_torch = # your code here

The function below will help with model training. Part of the code needs to be implemented independently.

In [ ]:
def train_model(
    model,
    opt,
    X_train_torch,
    y_train_torch,
    X_val_torch,
    y_val_torch,
    n_iterations=500,
    batch_size=32,
    show_plots=True,
    eval_every=50
):
    train_loss_history = []
    train_acc_history = []
    val_loss_history = []
    val_acc_history = []

    local_train_loss_history = []
    local_train_acc_history = []
    for i in range(n_iterations):

        # sample batch_size random observations
        ix = np.random.randint(0, len(X_train_torch), batch_size)
        x_batch = X_train_torch[ix]
        y_batch = y_train_torch[ix]

        # predict log-probabilities or logits
        y_predicted = # your code here
        
        # compute loss, just like before
        loss = # your code here

        # compute gradients
        # your code here

        # Adam step
        # your code here

        # clear gradients
        # your code here

        local_train_loss_history.append(loss.item())
        local_train_acc_history.append(
            accuracy_score(
                y_batch.to('cpu').detach().numpy(),
                y_predicted.to('cpu').detach().numpy().argmax(axis=1)
            )
        )

        if i % eval_every == 0:
            train_loss_history.append(np.mean(local_train_loss_history))
            train_acc_history.append(np.mean(local_train_acc_history))
            local_train_loss_history, local_train_acc_history = [], []

            predictions_val = model(X_val_torch)
            val_loss_history.append(loss_function(predictions_val, y_val_torch).to('cpu').detach().item())

            acc_score_val = accuracy_score(y_val_torch.cpu().numpy(), predictions_val.to('cpu').detach().numpy().argmax(axis=1))
            val_acc_history.append(acc_score_val)

            if show_plots:
                display.clear_output(wait=True)
                plot_train_process(train_loss_history, val_loss_history, train_acc_history, val_acc_history)
    return model

In [ ]:
bow_nn_model = train_model(
    model,
    opt,
    X_train_bow_torch,
    y_train_torch,
    X_test_bow_torch,
    y_test_torch,
    n_iterations=3000,
)

In [ ]:
# do not change the code in the block below
# __________start of block__________
out_dict = visualize_and_save_results(
    bow_nn_model,
    "bow_nn_torch",
    X_train_bow_torch,
    X_test_bow_torch,
    y_train,
    y_test,
    out_dict,
)

assert (
    out_dict["bow_log_reg_sklearn_test"] - out_dict["bow_nn_torch_test"] < 0.01
), "AUC ROC on test data should be close to the sklearn implementation"
# __________end of block__________

Now repeat the training procedure above, but for different values of `k` – the vocabulary size. In the `results` list, save the `AUC ROC` on the test part of the dataset for the model trained with a vocabulary of size `k`.

In [ ]:
vocab_sizes_list = np.arange(100, 5800, 700)
results = []

for k in vocab_sizes_list:
    # your code here
    predicted_probas_on_test_for_k_sized_dict = None
    assert predicted_probas_on_test_for_k_sized_dict is not None
    auc = roc_auc_score(y_test, predicted_probas_on_test_for_k_sized_dict)
    results.append(auc)

In [ ]:
# do not change the code in the block below
# __________start of block__________
assert len(results) == len(vocab_sizes_list), "Check the code above"
assert min(results) >= 0.65, "Seems like the model is not trained well enough"
assert results[-1] > 0.84, "Best AUC ROC should not be lower than 0.84"

plt.plot(vocab_sizes_list, results)
plt.xlabel("num of tokens")
plt.ylabel("AUC")
plt.grid()

out_dict["bow_k_vary"] = results
# __________end of block__________

### Task #2: Using TF-IDF Features.

To vectorize texts, you can also use TF-IDF. This allows you to exclude many words from consideration that do not have a significant impact when evaluating text dissimilarity.

You can read more about TF-IDF, for example, [here](https://towardsdatascience.com/tf-idf-for-document-ranking-from-scratch-in-python-on-real-world-dataset-796d339a4089).
You can also read about its manual implementation there.

Your task: vectorize the texts using TF-IDF (either using `TfidfVectorizer` from `sklearn` or by implementing it yourself) and build a classifier using PyTorch, similar to Task #1.

Then, also evaluate the classification quality by AUC ROC for different vocabulary sizes.

The classification quality should be at least 0.86 AUC ROC.

In [ ]:
# your code here

In [ ]:
X_train_tfidf = # your code here
X_test_tfidf = # your code here

In [ ]:
model = # your code here

In [ ]:
loss_function = # your code here
opt = # your code here

model_tf_idf = train_model(model, opt, X_train_tfidf_torch, y_train_torch, X_test_tfidf_torch, y_test_torch, n_iterations=3000)

In [ ]:
# do not change the code in the block below
# __________start of block__________
out_dict = visualize_and_save_results(
    model_tf_idf,
    "tf_idf_nn_torch",
    X_train_tfidf_torch,
    X_test_tfidf_torch,
    y_train,
    y_test,
    out_dict,
)

assert (
    out_dict["tf_idf_nn_torch_test"] >= out_dict["bow_nn_torch_test"]
), "AUC ROC on test data should be better or close to BoW for TF-iDF features"
# __________end of block__________

Similarly to Task #1, repeat the training procedure for different values of `k` – the vocabulary size, and save the `AUC ROC` on the test part of the dataset in the `results` list.

In [ ]:
vocab_sizes_list = np.arange(100, 5800, 700)
results = []

for k in vocab_sizes_list:
    # your code here
    predicted_probas_on_test_for_k_sized_dict = None
    assert predicted_probas_on_test_for_k_sized_dict is not None
    auc = roc_auc_score(y_test, predicted_probas_on_test_for_k_sized_dict)
    results.append(auc)

In [ ]:
# do not change the code in the block below
# __________start of block__________
assert len(results) == len(vocab_sizes_list), "Check the code above"
assert min(results) >= 0.65, "Seems like the model is not trained well enough"
assert results[-1] > 0.85, "Best AUC ROC for TF-iDF should not be lower than 0.84"

plt.plot(vocab_sizes_list, results)
plt.xlabel("num of tokens")
plt.ylabel("AUC")
plt.grid()

out_dict["tf_idf_k_vary"] = results
# __________end of block__________

### Task #3: Comparison with Naive Bayes Classifier.

Classical models are still capable of showing good results in many tasks. Train a Naive Bayes classifier on texts vectorized using BoW and TF-IDF and compare the results with the models above.

*Comment: please note that you need to choose an appropriate prior distribution for the features for this task, i.e., select the correct version of the classifier from `sklearn`: `GaussianNB`, `MultinomialNB`, `ComplementNB`, `BernoulliNB`, `CategoricalNB`*

In [ ]:
from sklearn.naive_bayes import # your code here

In [ ]:
clf_nb_bow = # your code here

# do not change the code in the block below
# __________start of block__________
out_dict = visualize_and_save_results(clf_nb_bow, 'bow_nb_sklearn', X_train_bow, X_test_bow, y_train, y_test, out_dict)
# __________end of block__________

In [ ]:
clf_nb_tfidf = # your code here

# do not change the code in the block below
# __________start of block__________
out_dict = visualize_and_save_results(clf_nb_tfidf, 'tf_idf_nb_sklearn', X_train_tfidf, X_test_tfidf, y_train, y_test, out_dict)
# __________end of block__________

In [ ]:
# do not change the code in the block below
# __________start of block__________
assert (
    out_dict["tf_idf_nb_sklearn_test"] > out_dict["bow_nb_sklearn_test"]
), " TF-iDF results should be better"
assert (
    out_dict["tf_idf_nb_sklearn_test"] > 0.86
), "TF-iDF Naive Bayes score should be above 0.86"
# __________end of block__________

### Task #4: Using Pretrained Embeddings
#### __Building Embeddings with word2vec__.
The approaches proposed above have significant drawbacks: they do not take into account the meaning of words when assigning a vector to each of them. Therefore, the distance between one-hot vectors for the words "cat" and "dog", for the words "cat" and "airplane", or for the words "cat" and "kitten" will be the same. For a person who knows the language, the difference between the words is obvious, as is the fact that "cat" and "kitten" are much closer to each other in meaning than "cat" and "airplane". When building an informative vector representation, we would also like to establish semantic similarity between words.

A simple idea (voiced in various forms many times) can help with this: __a word is significantly determined by the context in which it occurs__. Based on this, a simple conclusion can be made: some words are more typical for one context, while others are more typical for another. It is on this idea that word2vec is built (like many other embeddings).

From a word, one can learn to predict the context in which it occurs. Of course, the result will not be perfectly accurate. But if the model makes predictions better than chance, it means it is picking up some connection. And then the internal representation of the model for each word can be used as the desired vector representation, even in other tasks.

The formulation of the hypothesis above (the word is significantly connected with the context) allows using the entire set of texts for a chosen language as a training sample. Collected works of classics, encyclopedia articles, news reports – all this becomes a training sample. And the vector representations obtained on the basis of these texts allow capturing the connection between these words.
![](https://ruder.io/content/images/size/w2000/2016/04/word_embeddings_colah.png)
*Image source: https://ruder.io/word-embeddings-1/*

In this part, to obtain pretrained vector representations, we will use pretrained embeddings from the `gensim` library. Several embeddings pretrained on various text corpora are available in it. The full list can be found [here](https://radimrehurek.com/gensim/models/word2vec.html#pretrained-models). We remind you that it is better to use those embeddings that were trained on texts of a similar structure.

Your task: train a model (logistic regression or a two-layer neural network is sufficient) using the averaged embedding for all tokens in the review, achieve quality no worse than with BoW/TF-IDF, and reduce the degree of overfitting (the difference between AUC ROC on the training and test sets).

In [ ]:
import gensim.downloader as api
gensim_embedding_model = # your code here, e.g. api.load(...)

In [ ]:
def text_to_average_embedding(text, gensim_embedding_model):
    # your code here
    return embedding_for_text

In [ ]:
X_train_emb = [
    text_to_average_embedding(text, gensim_embedding_model) for text in texts_train
]
X_test_emb = [
    text_to_average_embedding(text, gensim_embedding_model) for text in texts_test
]

assert (
    len(X_train_emb[0]) == gensim_embedding_model.vector_size
), "Seems like the embedding shape is wrong"

In [ ]:
X_train_emb_torch = # your code here
X_test_emb_torch = # your code here

y_train_torch = # your code here
y_test_torch = # your code here

In [ ]:
model = # your code here

In [ ]:
loss_function = # your code here
opt = # your code here

model = train_model(model, opt, X_train_emb_torch, y_train_torch, X_test_emb_torch, y_test_torch, n_iterations=3000)

In [ ]:
# do not change the code in the block below
# __________start of block__________

out_dict = visualize_and_save_results(
    model,
    "emb_nn_torch",
    X_train_emb_torch,
    X_test_emb_torch,
    y_train,
    y_test,
    out_dict,
)
assert (
    out_dict["emb_nn_torch_test"] > 0.87
), "AUC ROC on test data should be better than 0.86"
assert (
    out_dict["emb_nn_torch_train"] - out_dict["emb_nn_torch_test"] < 0.1
), "AUC ROC on test and train data should not be different more than by 0.1"
# __________end of block__________

### Conclusions:
_Formulate conclusions about each of the word vectorization approaches, as well as the performance of the models (Naive Bayes, Logistic Regression, etc.)_

### Submission
Run the code below to generate the submission and submit the `submission_dict_hw_text_classification.json` file for review in the contest.

In [ ]:
# do not change the code in the block below
# __________start of block__________
FILENAME = "submission_dict_hw_text_classification.json"
with open(FILENAME, "w") as iofile:
    json.dump(out_dict, iofile)
print(f"File saved to `{FILENAME}`")
# __________end of block__________